# 🧠 Meta-Brain Prototype — PhenexAI
### Run cells 1 to 10 in order
Get API key free at: https://console.anthropic.com

In [ ]:
# CELL 1: PASTE YOUR API KEY HERE
API_KEY = 'sk-ant-PASTE-YOUR-KEY-HERE'
print('Key set. Run next cell.')

In [ ]:
# CELL 2: INSTALL
import subprocess
subprocess.run(['pip', 'install', 'anthropic', 'numpy', 'matplotlib', '-q'])
print('Done.')

In [ ]:
# CELL 3: API CONNECTION TEST
import anthropic
import json
import numpy as np
import matplotlib.pyplot as plt
import time

client = anthropic.Anthropic(api_key=API_KEY)

def call_claude(system_prompt, user_prompt, max_tokens=400):
    message = client.messages.create(
        model='claude-sonnet-4-20250514',
        max_tokens=max_tokens,
        system=system_prompt,
        messages=[{'role': 'user', 'content': user_prompt}]
    )
    return message.content[0].text

# Test
result = call_claude('You are helpful.', 'Say: API connected OK', 20)
print(result)

In [ ]:
# CELL 4: M1 WORLD MODEL
# Checks physical and logical consistency

def m1_world_model(question, answer):
    system = 'You are a physical consistency checker. Respond ONLY in JSON like this: {"score": 0.8, "reasoning": "one sentence"}. No markdown, no extra text.'
    prompt = 'Question: ' + question + '\nAnswer: ' + answer
    try:
        raw = call_claude(system, prompt, 100)
        raw = raw.strip().replace('```json', '').replace('```', '').strip()
        data = json.loads(raw)
        return float(data['score']), str(data['reasoning'])
    except Exception as e:
        return 0.5, str(e)

# Test
q = 'How fast does light travel?'
s1, r1 = m1_world_model(q, 'Light travels at 3x10^8 m/s in vacuum.')
s2, r2 = m1_world_model(q, 'Light travels at 10 km/h and can be stopped.')
print('M1 WORLD MODEL')
print('Good answer:', round(s1,3), '|', r1)
print('Bad answer: ', round(s2,3), '|', r2)

In [ ]:
# CELL 5: M5 HALLUCINATION CHECKER

def m5_sim_checker(question, answer):
    system = 'You are a hallucination detector. Respond ONLY in JSON like this: {"score": 0.8, "reasoning": "one sentence"}. No markdown, no extra text.'
    prompt = 'Question: ' + question + '\nAnswer: ' + answer
    try:
        raw = call_claude(system, prompt, 100)
        raw = raw.strip().replace('```json', '').replace('```', '').strip()
        data = json.loads(raw)
        return float(data['score']), str(data['reasoning'])
    except Exception as e:
        return 0.5, str(e)

# Test
s1, r1 = m5_sim_checker('When did Einstein publish relativity?', 'Einstein published special relativity in 1905.')
s2, r2 = m5_sim_checker('When did Einstein publish relativity?', 'Einstein won the Nobel Prize in 1905 for relativity.')
print('M5 HALLUCINATION CHECKER')
print('Real fact:      ', round(s1,3), '|', r1)
print('Hallucination:  ', round(s2,3), '|', r2)

In [ ]:
# CELL 6: M6 UNIVERSE MEMORY BOX
# Q(t) = 1 - exp(-lambda * t)

memory_store = []
memory_t = [0]
LAMBDA = 0.15

def m6_quality():
    return round(1.0 - np.exp(-LAMBDA * memory_t[0]), 4)

def m6_integrate(question, answer, score):
    memory_store.append({'q': question[:60], 'a': answer[:60], 's': round(score,3)})
    memory_t[0] += 1

def m6_memory_box(question, answer):
    if not memory_store:
        return 0.5, 'No memory yet'
    context = ' | '.join(['Q:' + m['q'] + ' S:' + str(m['s']) for m in memory_store[-3:]])
    system = 'You check memory consistency. Respond ONLY in JSON like this: {"score": 0.8, "reasoning": "one sentence"}. No markdown.'
    prompt = 'Memory: ' + context + '\nNew Q: ' + question + '\nNew A: ' + answer
    try:
        raw = call_claude(system, prompt, 100)
        raw = raw.strip().replace('```json', '').replace('```', '').strip()
        data = json.loads(raw)
        return float(data['score']), str(data['reasoning'])
    except Exception as e:
        return 0.5, str(e)

print('M6 MEMORY BOX')
print('Quality at t=0: ', m6_quality())
for i in range(5):
    m6_integrate('test question', 'test answer', 0.8)
print('Quality at t=5: ', m6_quality())
print('Quality at t=50:', round(1 - np.exp(-LAMBDA * 50), 4), '(projected)')
print('RAG at t=50:    ', round(np.exp(-0.001 * 50), 4), '(degrades)')

In [ ]:
# CELL 7: DECISION ENGINE + FULL META-BRAIN
# o* = argmax SUM w_i * p_i

WEIGHTS = {'M1': 0.40, 'M5': 0.35, 'M6': 0.25}

def generate_candidates(question, n=3):
    system = 'Generate ' + str(n) + ' different answers. Respond ONLY in JSON: {"answers": ["answer1", "answer2", "answer3"]}. No markdown.'
    try:
        raw = call_claude(system, question, 500)
        raw = raw.strip().replace('```json', '').replace('```', '').strip()
        return json.loads(raw)['answers'][:n]
    except:
        return [call_claude('Answer accurately.', question, 200)]

def metabrain(question):
    print('\n' + '='*50)
    print('QUESTION:', question)
    print('Memory quality:', m6_quality())

    candidates = generate_candidates(question, n=3)
    print('Generated', len(candidates), 'candidates')

    results = []
    for i, c in enumerate(candidates):
        s1, _ = m1_world_model(question, c)
        s5, _ = m5_sim_checker(question, c)
        s6, _ = m6_memory_box(question, c)
        final = WEIGHTS['M1']*s1 + WEIGHTS['M5']*s5 + WEIGHTS['M6']*s6
        results.append({'answer': c, 'M1': s1, 'M5': s5, 'M6': s6, 'final': final})
        print('Candidate', i+1, '| M1:', round(s1,2), 'M5:', round(s5,2), 'M6:', round(s6,2), '=> Final:', round(final,3))

    best = max(results, key=lambda x: x['final'])
    m6_integrate(question, best['answer'], best['final'])

    print('\nBEST ANSWER (score=' + str(round(best['final'],3)) + '):')
    print(best['answer'])
    print('Memory quality now:', m6_quality())
    return best

print('Meta-Brain ready. Run Cell 8 to test it.')

In [ ]:
# CELL 8: RUN META-BRAIN
# Change this question to anything you want

result = metabrain('What is the most important missing ingredient for AGI?')

In [ ]:
# CELL 9: META-BRAIN VS BASELINE

question = 'How does quantum entanglement work?'

print('BASELINE - single LLM:')
baseline = call_claude('Answer accurately.', question, 150)
print(baseline[:200])
print('No verification done.')

print('\nMETA-BRAIN - parallel modules + voting:')
mb = metabrain(question)
print('\nVerified by: M1 (physics) + M5 (hallucination) + M6 (memory)')
print('Final score:', round(mb['final'], 3))

In [ ]:
# CELL 10: MEMORY GROWTH CHART

more_questions = [
    'What is consciousness?',
    'Can a simulation share physics laws with its host universe?',
]

quality_log = [m6_quality()]
for q in more_questions:
    metabrain(q)
    quality_log.append(m6_quality())

t_range = np.arange(0, 80)
umb = [1 - np.exp(-LAMBDA * t) for t in t_range]
rag = [np.exp(-0.001 * t) for t in t_range]

plt.figure(figsize=(10, 4))
plt.plot(t_range, umb, color='green', linewidth=2, label='Memory Box Q(t) - improves')
plt.plot(t_range, rag, color='red', linewidth=2, linestyle='--', label='RAG P(t) - degrades')
plt.scatter(range(len(quality_log)), quality_log, s=120, color='gold', zorder=5, label='Your actual runs')
plt.xlabel('Runs')
plt.ylabel('Quality')
plt.title('M6: Memory Box grows. RAG degrades.')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Final memory quality:', m6_quality())
print('Total runs:', memory_t[0])
print('Prototype complete.')